In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors

import torch

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor  = torch.tensor(X_test, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor  = torch.tensor(y_test, dtype=torch.float32)



In [ ]:
# 2. Create TensorDataset objects

from torch.utils.data import TensorDataset

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset  = TensorDataset(X_test_tensor, y_test_tensor)



In [ ]:
# 3. Create DataLoaders
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)



In [ ]:
# 4. Print shape of one batch

images, labels = next(iter(train_loader))
print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)


In [ ]:
# 5. Display sample images

import matplotlib.pyplot as plt

# Convert first 6 images back to HWC format for plotting
plt.figure(figsize=(10, 6))

for i in range(6):
    img = images[i].permute(1, 2, 0).numpy()  # C,H,W → H,W,C
    plt.subplot(2, 3, i+1)
    plt.imshow(img)
    plt.title(f"Age: {labels[i].item():.0f}")
    plt.axis("off")

plt.show()


In [ ]:
# Task 1: Write your model class here:

class AgeRegressor(nn.Module):
    def __init__(self, input_dim):
        super(AgeRegressor, self).__init__()

        self.fc1 = nn.Linear(input_dim, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, 1)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = self.fc4(x)
        return x



In [ ]:
# Task 2: Write your training loop here:
def train_one_epoch(model, loader, loss_fn, optimizer, device):
    model.train()
    total_loss = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images).squeeze()
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


In [ ]:
# Task 3: Write your validation loop here:
def validate(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images).squeeze()
            loss = loss_fn(outputs, labels)
            total_loss += loss.item()

    return total_loss / len(loader)



In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Determine input dimension automatically
sample_batch, _ = next(iter(train_loader))
input_dim = sample_batch.view(sample_batch.size(0), -1).shape[1]

model = AgeRegressor(input_dim).to(device)

loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


In [ ]:
# Train for 20 epochs and track losses
#x = F.relu(self)

train_losses = []
val_losses = []

epochs = 20

for epoch in range(epochs):
    train_loss = train_one_epoch(model, train_loader, loss_fn, optimizer, device)
    val_loss = validate(model, test_loader, loss_fn, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")


In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))
plt.plot(train_losses, label="Training Loss", marker='o')
plt.plot(val_losses, label="Validation Loss", marker='o')
plt.title("Training vs Validation Loss Over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Task 2 (Bonus): Write your code here:
import matplotlib.pyplot as plt
import torch

model.eval()

images, labels = next(iter(test_loader))
images, labels = images.to(device), labels.to(device)

with torch.no_grad():
    preds = model(images).squeeze()

# Move to CPU for plotting
images = images.cpu()
labels = labels.cpu()
preds = preds.cpu()

plt.figure(figsize=(12, 8))

for i in range(6):
    img = images[i].permute(1, 2, 0).numpy()  # C,H,W → H,W,C
    plt.subplot(2, 3, i+1)
    plt.imshow(img)
    plt.title(f"Pred: {preds[i].item():.1f} | Actual: {labels[i].item():.1f}")
    plt.axis("off")

plt.tight_layout()
plt.show()
